# 高温作业专用服装设计 — 增强模型

基于 Cobb-Douglas 效用函数的综合优化模型，同时考虑**舒适度**与**隔热性能**。

与原模型的区别：
- **原模型**：仅以最小厚度/最短平衡时间为目标，满足安全约束即可
- **增强模型**：在安全约束基础上，最大化舒适度与隔热性能的加权效用

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize, minimize_scalar

from model import (
    load_appendix_xlsx,
    solve_heat_cn,
    steady_state_h_in,
    rmse,
    write_simple_xlsx,
)

# 全局配置
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

ROOT = Path('..').resolve()
APPENDIX = ROOT / 'CUMCM-2018-Problem-A-Chinese-Appendix.xlsx'

# 步长配置（平衡速度与精度）
DX_MM = 0.2
DT_S = 2.0

In [ ]:
# 加载数据 & 运行问题一反演（复用已有结果）
data = load_appendix_xlsx(APPENDIX)
materials = data.materials
thickness_p1 = {'I': 0.6, 'II': 6.0, 'III': 3.6, 'IV': 5.0}
t_skin = 37.0
t_surface = float(data.measured_temp_c[-1])

# 问题一：快速反演 h_out, h_in
best_h = {'h_out': None, 'h_in': None, 'rmse': float('inf')}
for h_out in np.arange(100.0, 120.01, 0.5):
    h_in = steady_state_h_in(materials, thickness_p1, 75.0, t_skin, t_surface, h_out)
    times_p1, temps_p1 = solve_heat_cn(
        materials, thickness_p1, 75.0, h_out, h_in, t_skin,
        total_time_s=5400, dx_mm=DX_MM, dt_s=DT_S, store_all=False
    )
    target = np.interp(times_p1, data.measured_time_s, data.measured_temp_c)
    score = rmse(temps_p1[:, 0], target)
    if score < best_h['rmse']:
        best_h = {'h_out': h_out, 'h_in': h_in, 'rmse': score}

H_OUT = float(best_h['h_out'])
H_IN = float(best_h['h_in'])
print(f'Problem 1: h_I={H_OUT:.1f}, h_IV={H_IN:.4f}, RMSE={best_h["rmse"]:.4f}')

---
## 问题二：增强模型（单变量效用优化）

### 模型定义

**舒适度指数**（平方根递减）：
$$C(d_2) = 1 - \sqrt{\frac{d_2 - d_{2,\min}}{d_{2,\max} - d_{2,\min}}}$$

**隔热性能指数**（热阻归一化）：
$$\gamma_2 = \frac{d_2}{k_2}, \quad R(d_2) = \frac{\gamma_2 - \gamma_{2,\min}}{\gamma_{2,\max} - \gamma_{2,\min}}$$

**Cobb-Douglas 效用函数**：
$$U(d_2) = C^\alpha \cdot R^{1-\alpha}$$

**临界厚度**：二分法求解 $D_1$（最高温=47°C）和 $D_2$（55min时=44°C），$\tilde{d}_{2,\min} = \max(D_1, D_2)$

In [ ]:
# ===== 问题二参数 =====
T_ENV_P2 = 65.0
TOTAL_TIME_P2 = 3600  # 60 min
D2_MIN, D2_MAX = 0.6, 25.0  # mm
K2 = materials['II']['k']  # 热导率

# 热阻
gamma_min = D2_MIN / K2
gamma_max = D2_MAX / K2

def run_skin_temp(d2, t_env=T_ENV_P2, total_time_s=TOTAL_TIME_P2):
    """运行 PDE 并返回皮肤温度时间序列"""
    thickness = {'I': 0.6, 'II': d2, 'III': 3.6, 'IV': 5.5}
    _, temps = solve_heat_cn(
        materials, thickness, t_env, H_OUT, H_IN, t_skin,
        total_time_s=total_time_s, dx_mm=DX_MM, dt_s=DT_S, store_all=False
    )
    return temps[:, 0]

def max_temp(d2):
    """皮肤外侧最高温度"""
    return float(np.max(run_skin_temp(d2)))

def temp_at_time(d2, target_time_s):
    """指定时刻的皮肤温度"""
    skin = run_skin_temp(d2)
    idx = min(int(target_time_s / DT_S), len(skin) - 1)
    return float(skin[idx])

In [ ]:
# ===== 二分法求临界厚度 =====
def bisection_critical(f_target, target_val, a, b, tol=0.01, max_iter=50):
    """
    二分法求解 f(d) = target_val
    假设 f(d) 关于 d 单调递减（越厚越凉）
    若 target 不在 [f(b), f(a)] 内，说明该约束恒满足或恒不满足，返回 None
    """
    fa = f_target(a)
    fb = f_target(b)

    if not (fb <= target_val <= fa):
        # 若 fa < target_val，约束恒满足（即使最薄也满足），返回下限
        # 若 fb > target_val，约束恒不满足（即使最厚也超标），返回上限
        return None

    for _ in range(max_iter):
        m = (a + b) / 2.0
        fm = f_target(m)
        if abs(fm - target_val) < tol:
            return m
        if fm > target_val:
            a = m
        else:
            b = m
    return (a + b) / 2.0

# D1: max_temp(d2) = 47
D1 = bisection_critical(max_temp, 47.0, D2_MIN, D2_MAX)

# D2: temp_at_55min(d2) = 44
TARGET_TIME_55MIN = 55 * 60  # 3300 s
D2 = bisection_critical(lambda d: temp_at_time(d, TARGET_TIME_55MIN), 44.0, D2_MIN, D2_MAX)

print(f'Critical thickness D1 (max_T=47°C):  {D1:.2f} mm' if D1 else 'D1: constraint always satisfied (max_T < 47°C for all d2)')
print(f'Critical thickness D2 (T_55min=44°C): {D2:.2f} mm' if D2 else 'D2: not found')
# 若某约束恒满足，取其下限；若恒不满足，取其上限
d2_critical = max(D1 if D1 is not None else D2_MIN, D2 if D2 is not None else D2_MIN)
print(f'Safety lower bound d2_min = max(D1, D2) = {d2_critical:.2f} mm')

In [ ]:
# ===== 效用函数 & 优化 =====
ALPHA = 0.5  # 舒适度权重（可调）

def comfort(d2):
    x = (d2 - D2_MIN) / (D2_MAX - D2_MIN)
    return 1.0 - np.sqrt(np.clip(x, 0.0, 1.0))

def insulation(d2):
    gamma = d2 / K2
    x = (gamma - gamma_min) / (gamma_max - gamma_min)
    return np.clip(x, 0.0, 1.0)

def utility(d2):
    c = comfort(d2)
    r = insulation(d2)
    return (c ** ALPHA) * (r ** (1.0 - ALPHA))

# 在安全可行域内搜索最大化 U
d2_search = np.linspace(d2_critical, D2_MAX, 500)
u_values = utility(d2_search)
best_idx = np.argmax(u_values)
d2_opt = d2_search[best_idx]
u_opt = u_values[best_idx]

C_opt = comfort(d2_opt)
R_opt = insulation(d2_opt)

print(f'=== 问题二 增强模型结果 ===')
print(f'Critical safe thickness: {d2_critical:.2f} mm')
print(f'Optimal thickness d2*:    {d2_opt:.2f} mm')
print(f'  Comfort C:  {C_opt:.4f}')
print(f'  Insulation R: {R_opt:.4f}')
print(f'  Utility U:    {u_opt:.4f}')

# 验证安全约束
skin_opt = run_skin_temp(d2_opt)
max_t = float(np.max(skin_opt))
over_44 = float(np.sum(skin_opt > 44.0)) * DT_S
print(f'\nVerification:')
print(f'  max skin temp: {max_t:.2f}°C (limit: 47°C)')
print(f'  time >44°C:    {over_44:.0f}s (limit: 300s)')

---
## 问题二：可视化

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

d2_full = np.linspace(D2_MIN, D2_MAX, 500)
d2_feasible = np.linspace(d2_critical, D2_MAX, 200)

# --- C(d2) ---
ax = axes[0, 0]
ax.plot(d2_full, comfort(d2_full), 'b-', linewidth=1.5, alpha=0.3, label='Infeasible')
ax.plot(d2_feasible, comfort(d2_feasible), 'b-', linewidth=2.5, label='Feasible')
ax.axvline(d2_critical, color='red', linestyle='--', linewidth=2,
           label=f'Critical d2 = {d2_critical:.1f} mm')
ax.set_xlabel('d2 (mm)'); ax.set_ylabel('C')
ax.set_title('Comfort Index C(d2)')
ax.legend(); ax.set_ylim(0, 1.05); ax.grid(True, alpha=0.3)

# --- R(d2) ---
ax = axes[0, 1]
ax.plot(d2_full, insulation(d2_full), 'r-', linewidth=1.5, alpha=0.3)
ax.plot(d2_feasible, insulation(d2_feasible), 'r-', linewidth=2.5)
ax.axvline(d2_critical, color='red', linestyle='--', linewidth=2)
ax.set_xlabel('d2 (mm)'); ax.set_ylabel('R')
ax.set_title('Insulation Index R(d2)')
ax.set_ylim(0, 1.05); ax.grid(True, alpha=0.3)

# --- U(d2) --- FIXED: feasible vs infeasible
ax = axes[1, 0]
ax.plot(d2_full, utility(d2_full), 'purple', linewidth=1.5, alpha=0.3, label='Infeasible')
ax.plot(d2_feasible, utility(d2_feasible), 'purple', linewidth=2.5, label='Feasible')
ax.axvline(d2_critical, color='red', linestyle='--', linewidth=2,
           label=f'Critical d2 = {d2_critical:.1f} mm')
d2_global = d2_full[np.argmax(utility(d2_full))]
u_global = np.max(utility(d2_full))
ax.plot(d2_global, u_global, 'o', color='gray', markersize=12, alpha=0.6,
        label=f'Global max (infeasible): d2={d2_global:.1f}mm')
ax.plot(d2_opt, u_opt, 'go', markersize=12,
        label=f'Constrained max: d2={d2_opt:.1f}mm, U={u_opt:.3f}')
ax.axvspan(0, d2_critical, alpha=0.08, color='red')
ax.set_xlabel('d2 (mm)'); ax.set_ylabel('U')
ax.set_title(f'Utility U = C^{ALPHA} * R^(1-{ALPHA})')
ax.legend(fontsize=8); ax.set_ylim(0, 0.45); ax.grid(True, alpha=0.3)

# --- C vs R trade-off ---
ax = axes[1, 1]
c_full = comfort(d2_full)
r_full = insulation(d2_full)
c_feas = comfort(d2_feasible)
r_feas = insulation(d2_feasible)
ax.scatter(c_full, r_full, c=utility(d2_full), cmap='plasma', s=3, alpha=0.15)
ax.scatter(c_feas, r_feas, c=utility(d2_feasible), cmap='plasma', s=20, alpha=0.9)
ax.plot(C_opt, R_opt, 'go', markersize=14, label=f'Optimal (d2={d2_opt:.1f})')
ax.plot(comfort(d2_global), insulation(d2_global), 'o', color='gray',
        markersize=12, alpha=0.5, label=f'Global (infeasible, d2={d2_global:.1f})')
ax.set_xlabel('Comfort C'); ax.set_ylabel('Insulation R')
ax.set_title('C-R Trade-off (highlight = feasible)')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle('Enhanced Model: Problem 2 Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('problem2_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ===== alpha 敏感性分析 =====
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

alphas = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(alphas)))
optimal_d2 = []

for alpha, color in zip(alphas, colors):
    def u_alpha(d):
        c = np.clip(comfort(d), 0, 1)
        r = np.clip(insulation(d), 0, 1)
        return (c ** alpha) * (r ** (1 - alpha))

    u_vals = u_alpha(d2_full)
    d2_best = d2_full[np.argmax(u_vals)]
    optimal_d2.append((alpha, d2_best))

    axes[0].plot(d2_full, u_vals, color=color, linewidth=2, label=f'alpha={alpha}')

axes[0].set_xlabel('d2 (mm)')
axes[0].set_ylabel('U')
axes[0].set_title('Utility for Different alpha')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].plot(d2_full, comfort(d2_full), 'b-', linewidth=2, label='C(d2)')
axes[1].plot(d2_full, insulation(d2_full), 'r-', linewidth=2, label='R(d2)')
axes[1].axvline(d2_critical, color='gray', linestyle='--', alpha=0.5, label=f'Critical={d2_critical:.1f}')
axes[1].set_xlabel('d2 (mm)')
axes[1].set_ylabel('Index')
axes[1].set_title('C(d2) and R(d2)')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

alphas_list = [a for a, _ in optimal_d2]
d2_list = [d for _, d in optimal_d2]
axes[2].bar(range(len(alphas_list)), d2_list, tick_label=[f'{a:.1f}' for a in alphas_list], color=colors)
axes[2].axhline(d2_critical, color='red', linestyle='--', alpha=0.5, label=f'd2_critical={d2_critical:.1f}')
axes[2].set_xlabel('alpha')
axes[2].set_ylabel('Optimal d2 (mm)')
axes[2].set_title('Optimal d2 vs alpha')
axes[2].legend(fontsize=8)

plt.suptitle('Sensitivity Analysis: alpha parameter', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('alpha_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 问题三：增强模型（二维效用优化）

### 模型定义

**舒适度指数**：
$$C(d_2, d_4) = 1 - \sqrt{\frac{(d_2+d_4) - (d_{2,\min}+d_{4,\min})}{(d_{2,\max}+d_{4,\max}) - (d_{2,\min}+d_{4,\min})}}$$

**隔热性能指数**（总热阻归一化）：
$$R(d_2, d_4) = \frac{(\gamma_2+\gamma_4) - (\gamma_{2,\min}+\gamma_{4,\min})}{(\gamma_{2,\max}+\gamma_{4,\max}) - (\gamma_{2,\min}+\gamma_{4,\min})}$$

**效用函数**：
$$U = C^\alpha \cdot R^{1-\alpha}$$

**求解**：先网格遍历找可行域，再用 SLSQP 在可行域内精细化搜索

In [ ]:
# ===== 问题三参数 =====
T_ENV_P3 = 80.0
TOTAL_TIME_P3 = 1800  # 30 min
D4_MIN, D4_MAX = 0.6, 6.4  # mm
K4 = materials['IV']['k']

# 总热阻范围
gamma2_min = D2_MIN / K2
gamma2_max = D2_MAX / K2
gamma4_min = D4_MIN / K4
gamma4_max = D4_MAX / K4
gamma_sum_min = gamma2_min + gamma4_min
gamma_sum_max = gamma2_max + gamma4_max
d_sum_min = D2_MIN + D4_MIN
d_sum_max = D2_MAX + D4_MAX

def run_skin_temp_p3(d2, d4):
    thickness = {'I': 0.6, 'II': d2, 'III': 3.6, 'IV': d4}
    _, temps = solve_heat_cn(
        materials, thickness, T_ENV_P3, H_OUT, H_IN, t_skin,
        total_time_s=TOTAL_TIME_P3, dx_mm=DX_MM, dt_s=DT_S, store_all=False
    )
    return temps[:, 0]

def comfort_p3(d2, d4):
    x = (d2 + d4 - d_sum_min) / (d_sum_max - d_sum_min)
    return 1.0 - np.sqrt(np.clip(x, 0.0, 1.0))

def insulation_p3(d2, d4):
    gamma_sum = d2 / K2 + d4 / K4
    x = (gamma_sum - gamma_sum_min) / (gamma_sum_max - gamma_sum_min)
    return np.clip(x, 0.0, 1.0)

def utility_p3(d2, d4, alpha=ALPHA):
    c = comfort_p3(d2, d4)
    r = insulation_p3(d2, d4)
    if c <= 0 or r <= 0:
        return 0.0
    return (c ** alpha) * (r ** (1.0 - alpha))

In [ ]:
# ===== 网格遍历：构建可行域 =====
print('Grid search for feasible region...')
d2_grid = np.arange(18.0, 21.01, 0.2)
d4_grid = np.arange(0.6, 6.41, 0.2)

feasible_p3 = []
total = len(d2_grid) * len(d4_grid)
count = 0

for d2 in d2_grid:
    for d4 in d4_grid:
        count += 1
        skin = run_skin_temp_p3(d2, d4)
        max_t = float(np.max(skin))
        over_44 = float(np.sum(skin > 44.0)) * DT_S
        if max_t <= 47.0 and over_44 <= 300.0:
            u = utility_p3(d2, d4)
            feasible_p3.append({
                'd2': d2, 'd4': d4,
                'max_T': max_t, 'over_44_s': over_44,
                'C': comfort_p3(d2, d4),
                'R': insulation_p3(d2, d4),
                'U': u
            })
    if count % 50 == 0:
        print(f'  Progress: {count}/{total}')

print(f'Feasible points found: {len(feasible_p3)}')

# 找最大 U
best_by_u = max(feasible_p3, key=lambda r: r['U'])
print(f'\nBest by max U:')
print(f'  d2={best_by_u["d2"]:.1f} mm, d4={best_by_u["d4"]:.1f} mm')
print(f'  C={best_by_u["C"]:.4f}, R={best_by_u["R"]:.4f}, U={best_by_u["U"]:.4f}')
print(f'  max_T={best_by_u["max_T"]:.2f}°C, over_44={best_by_u["over_44_s"]:.0f}s')

In [ ]:
# ===== SLSQP 精细化优化 =====
def objective(x):
    """最小化负效用（SLSQP 默认最小化）"""
    return -utility_p3(x[0], x[1])

def constraint_max_temp(x):
    """max_T <= 47"""
    skin = run_skin_temp_p3(x[0], x[1])
    return 47.0 - float(np.max(skin))

def constraint_over_44(x):
    """over_44_time <= 300s"""
    skin = run_skin_temp_p3(x[0], x[1])
    over = float(np.sum(skin > 44.0)) * DT_S
    return 300.0 - over

constraints = [
    {'type': 'ineq', 'fun': constraint_max_temp},
    {'type': 'ineq', 'fun': constraint_over_44},
]
bounds = [(D2_MIN, D2_MAX), (D4_MIN, D4_MAX)]

# 从网格最优点出发
x0 = np.array([best_by_u['d2'], best_by_u['d4']])
print(f'SLSQP starting point: d2={x0[0]:.2f}, d4={x0[1]:.2f}')

result = minimize(
    objective, x0,
    method='SLSQP',
    bounds=bounds,
    constraints=constraints,
    options={'ftol': 1e-6, 'maxiter': 50, 'disp': True}
)

if result.success:
    d2_slsqp, d4_slsqp = result.x
    print(f'\nSLSQP optimal:')
    print(f'  d2={d2_slsqp:.2f} mm, d4={d4_slsqp:.2f} mm')
    print(f'  U={utility_p3(d2_slsqp, d4_slsqp):.4f}')
else:
    print(f'SLSQP failed: {result.message}')
    d2_slsqp, d4_slsqp = x0

---
## 问题三：可视化

In [ ]:
# ===== 可行域 & U 等值线图 =====
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 构造 U 矩阵
U_matrix = np.zeros((len(d2_grid), len(d4_grid)))
feasible_mask = np.zeros((len(d2_grid), len(d4_grid)), dtype=bool)

for i, d2 in enumerate(d2_grid):
    for j, d4 in enumerate(d4_grid):
        U_matrix[i, j] = utility_p3(d2, d4)

for r in feasible_p3:
    i = np.argmin(np.abs(d2_grid - r['d2']))
    j = np.argmin(np.abs(d4_grid - r['d4']))
    feasible_mask[i, j] = True

D4_mesh, D2_mesh = np.meshgrid(d4_grid, d2_grid)

# --- U contour + feasible region ---
ax = axes[0]
contour = ax.contourf(D2_mesh, D4_mesh, U_matrix, levels=20, cmap='viridis', alpha=0.9)
feasible_d2 = [r['d2'] for r in feasible_p3]
feasible_d4 = [r['d4'] for r in feasible_p3]
ax.scatter(feasible_d2, feasible_d4, c='red', s=2, alpha=0.5, label='Feasible')
ax.plot(best_by_u['d2'], best_by_u['d4'], 'r*', markersize=15, label=f'Max U ({best_by_u["d2"]:.1f}, {best_by_u["d4"]:.1f})')
ax.plot(d2_slsqp, d4_slsqp, 'y*', markersize=15, markerfacecolor='yellow', markeredgecolor='black', label=f'SLSQP ({d2_slsqp:.1f}, {d4_slsqp:.1f})')
ax.set_xlabel('d2 (mm)')
ax.set_ylabel('d4 (mm)')
ax.set_title('Utility U(d2, d4) with Feasible Region')
ax.legend(fontsize=8)
plt.colorbar(contour, ax=ax, label='U')

# --- C-R Pareto 前沿 ---
ax = axes[1]
c_vals = np.array([r['C'] for r in feasible_p3])
r_vals = np.array([r['R'] for r in feasible_p3])
u_vals = np.array([r['U'] for r in feasible_p3])
sc = ax.scatter(c_vals, r_vals, c=u_vals, cmap='plasma', s=30, alpha=0.8, edgecolors='none')

# Pareto frontier (max R for each level of C)
c_bins = np.linspace(c_vals.min(), c_vals.max(), 20)
pareto_c, pareto_r = [], []
for i in range(len(c_bins) - 1):
    mask = (c_vals >= c_bins[i]) & (c_vals < c_bins[i + 1])
    if mask.sum() > 0:
        best_idx = np.argmax(u_vals[mask])
        idx = np.where(mask)[0][best_idx]
        pareto_c.append(c_vals[idx])
        pareto_r.append(r_vals[idx])

ax.plot(pareto_c, pareto_r, 'r-', linewidth=2, label='Pareto frontier')
ax.plot(best_by_u['C'], best_by_u['R'], 'r*', markersize=15, label=f'Max U')
ax.set_xlabel('Comfort C')
ax.set_ylabel('Insulation R')
ax.set_title('C-R Trade-off (Pareto Frontier)')
ax.legend(fontsize=8)
plt.colorbar(sc, ax=ax, label='U')

plt.suptitle('Enhanced Model: Problem 3 Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('problem3_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ===== 原模型 vs 增强模型对比图 =====
fig, ax = plt.subplots(figsize=(10, 6))

# 原模型结果（来自 repro_notebook）
original = {
    'P2_d2_original': 17.8,    # 原模型: 最小可行厚度
    'P3_d2_original': 19.3,    # 原模型 (paper match)
    'P3_d4_original': 6.4,
}

models = ['Original', 'Enhanced']
p2_values = [17.8, d2_opt]
p3_d2_values = [19.3, d2_slsqp]
p3_d4_values = [6.4, d4_slsqp]

x = np.arange(len(models))
width = 0.25

bars1 = ax.bar(x - width, p2_values, width, label='P2: d2 (mm)', color='steelblue')
bars2 = ax.bar(x, p3_d2_values, width, label='P3: d2 (mm)', color='coral')
bars3 = ax.bar(x + width, p3_d4_values, width, label='P3: d4 (mm)', color='seagreen')

# 标注数值
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
for bar in bars3:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylabel('Thickness (mm)')
ax.set_title('Model Comparison: Original vs Enhanced')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ===== 厚度-温度关系图：验证临界厚度的物理意义 =====
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Problem 2
d2_test = np.arange(D2_MIN + 1, D2_MAX + 1, 2)
ax = axes[0]
for d2 in d2_test:
    skin = run_skin_temp(d2, T_ENV_P2, TOTAL_TIME_P2)
    time_axis = np.arange(len(skin)) * DT_S / 60
    ax.plot(time_axis, skin, linewidth=1, alpha=0.7, label=f'd2={d2:.0f}')

# 标注临界线
ax.axhline(47, color='red', linestyle='--', alpha=0.8, label='47°C limit')
ax.axhline(44, color='orange', linestyle='--', alpha=0.8, label='44°C threshold')
ax.axvline(55, color='orange', linestyle=':', alpha=0.8, label='t=55min')
ax.set_xlabel('Time (min)')
ax.set_ylabel('Skin Temperature (°C)')
ax.set_title('Problem 2: Skin Temp vs Time (various d2)')
ax.legend(fontsize=7, ncol=2)
ax.grid(True, alpha=0.3)

# Problem 3: sweep d2 at fixed d4=6.4
d2_test_p3 = np.arange(18.0, 22.0, 1)
ax = axes[1]
for d2 in d2_test_p3:
    skin = run_skin_temp_p3(d2, 6.4)
    time_axis = np.arange(len(skin)) * DT_S / 60
    ax.plot(time_axis, skin, linewidth=1, alpha=0.7, label=f'd2={d2:.1f}, d4=6.4')

ax.axhline(47, color='red', linestyle='--', alpha=0.8, label='47°C limit')
ax.axhline(44, color='orange', linestyle='--', alpha=0.8)
ax.axvline(25, color='orange', linestyle=':', alpha=0.8, label=f't=25min')
ax.set_xlabel('Time (min)')
ax.set_ylabel('Skin Temperature (°C)')
ax.set_title('Problem 3: Skin Temp vs Time (various d2, d4=6.4)')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

plt.suptitle('Temperature Evolution Verification', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('temperature_evolution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('All figures saved to enhanced_model/')